In [1]:
# ============================================================
# FINAL MODELS — CALIBRATION / BRIER SUMMARY
# PCA TRAIN-ONLY FINAL VERSION
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score,
    average_precision_score,
    log_loss
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PREDICTION_FILES = {
    "victimization": Path("./final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv"),
    "perpetration": Path("./final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv"),
    "overlap": Path(
        "./final_overlap/overlap_final/"
        "overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42/"
        "outputs/predictions_with_probs.csv"
    ),
}

OUTPUT_DIR = Path("./final_calibration_brier_PCA_trainonly")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_BINS = 10


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def detect_col(df, candidates):
    cols_lower = {str(c).lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]

    for c in df.columns:
        cl = str(c).lower()
        if any(cand.lower() in cl for cand in candidates):
            return c

    return None


def load_prediction_file(path, outcome):
    df = pd.read_csv(path)

    y_true_col = detect_col(df, [
        f"y_true_{outcome}",
        "y_true_victim",
        "y_true_perp",
        "y_true_perpetration",
        "y_true_overlap",
        "y_true_intersect",
        "y_true",
        "target"
    ])

    y_pred_col = detect_col(df, [
        f"y_pred_{outcome}",
        "y_pred_victim",
        "y_pred_perp",
        "y_pred_perpetration",
        "y_pred_overlap",
        "y_pred_intersect",
        "y_pred",
        "prediction"
    ])

    y_prob_col = detect_col(df, [
        f"y_prob_{outcome}",
        "y_prob_victim",
        "y_prob_perp",
        "y_prob_perpetration",
        "y_prob_overlap",
        "y_prob_intersect",
        "y_prob",
        "probability",
        "score"
    ])

    if y_true_col is None or y_pred_col is None or y_prob_col is None:
        raise ValueError(
            f"Could not detect required columns for {outcome}. "
            f"Columns found: {list(df.columns)}"
        )

    out = pd.DataFrame({
        "y_true": pd.to_numeric(df[y_true_col], errors="coerce"),
        "y_pred": pd.to_numeric(df[y_pred_col], errors="coerce"),
        "y_prob": pd.to_numeric(df[y_prob_col], errors="coerce"),
    }).dropna()

    out["y_true"] = (out["y_true"] > 0).astype(int)
    out["y_pred"] = (out["y_pred"] > 0).astype(int)

    # Clip probabilities for log loss safety
    out["y_prob"] = out["y_prob"].clip(1e-6, 1 - 1e-6)

    return out


def calibration_bins(df, outcome, n_bins=10):
    d = df.copy()

    # Equal-width bins 0-1
    d["prob_bin"] = pd.cut(
        d["y_prob"],
        bins=np.linspace(0, 1, n_bins + 1),
        include_lowest=True,
        duplicates="drop"
    )

    rows = []

    for bin_label, sub in d.groupby("prob_bin", observed=False):
        if len(sub) == 0:
            continue

        rows.append({
            "outcome": outcome,
            "bin": str(bin_label),
            "n": int(len(sub)),
            "mean_predicted_probability": float(sub["y_prob"].mean()),
            "observed_event_rate": float(sub["y_true"].mean()),
            "absolute_calibration_error": float(abs(sub["y_prob"].mean() - sub["y_true"].mean())),
            "positives": int(sub["y_true"].sum()),
            "negatives": int((sub["y_true"] == 0).sum()),
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Run calibration/Brier
# ------------------------------------------------------------

summary_rows = []
all_bins = []

for outcome, path in PREDICTION_FILES.items():
    print("\n======================================")
    print("Outcome:", outcome)
    print("Path:", path)
    print("Exists:", path.exists())

    dfp = load_prediction_file(path, outcome)

    y_true = dfp["y_true"].to_numpy()
    y_prob = dfp["y_prob"].to_numpy()
    y_pred = dfp["y_pred"].to_numpy()

    prevalence = y_true.mean()

    brier = brier_score_loss(y_true, y_prob)

    try:
        auc_roc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc_roc = np.nan

    try:
        auc_pr = average_precision_score(y_true, y_prob)
    except Exception:
        auc_pr = np.nan

    try:
        ll = log_loss(y_true, y_prob, labels=[0, 1])
    except Exception:
        ll = np.nan

    mean_prob_pos = y_prob[y_true == 1].mean()
    mean_prob_neg = y_prob[y_true == 0].mean()

    # Brier skill score against prevalence-only model
    # Reference Brier = prevalence * (1 - prevalence)
    brier_ref = prevalence * (1 - prevalence)
    brier_skill = 1 - (brier / brier_ref) if brier_ref > 0 else np.nan

    summary_rows.append({
        "outcome": outcome,
        "n": int(len(y_true)),
        "positives": int(y_true.sum()),
        "negatives": int((y_true == 0).sum()),
        "prevalence": prevalence,
        "mean_predicted_probability": float(y_prob.mean()),
        "mean_probability_positives": float(mean_prob_pos),
        "mean_probability_negatives": float(mean_prob_neg),
        "brier_score": float(brier),
        "brier_reference_prevalence_model": float(brier_ref),
        "brier_skill_score": float(brier_skill),
        "roc_auc": float(auc_roc),
        "average_precision_pr_auc": float(auc_pr),
        "log_loss": float(ll),
    })

    bins_df = calibration_bins(dfp, outcome, n_bins=N_BINS)
    all_bins.append(bins_df)

    print("n:", len(y_true))
    print("prevalence:", round(prevalence, 3))
    print("mean predicted probability:", round(y_prob.mean(), 3))
    print("Brier:", round(brier, 4))
    print("Brier skill:", round(brier_skill, 4))
    print("ROC AUC:", round(auc_roc, 4))
    print("PR AUC:", round(auc_pr, 4))


summary_df = pd.DataFrame(summary_rows)
bins_all_df = pd.concat(all_bins, ignore_index=True)

summary_df.to_csv(OUTPUT_DIR / "final_models_brier_calibration_summary.csv", index=False)
bins_all_df.to_csv(OUTPUT_DIR / "final_models_calibration_bins.csv", index=False)

print("\n=== BRIER / CALIBRATION SUMMARY ===")
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n=== CALIBRATION BINS ===")
print(bins_all_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nSaved to:")
print(OUTPUT_DIR.resolve())


Outcome: victimization
Path: final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv
Exists: True
n: 942
prevalence: 0.494
mean predicted probability: 0.497
Brier: 0.2307
Brier skill: 0.0771
ROC AUC: 0.6342
PR AUC: 0.5881

Outcome: perpetration
Path: final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv
Exists: True
n: 942
prevalence: 0.235
mean predicted probability: 0.575
Brier: 0.2802
Brier skill: -0.5607
ROC AUC: 0.6951
PR AUC: 0.3985

Outcome: overlap
Path: final_overlap/overlap_final/overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42/outputs/predictions_with_probs.csv
Exists: True
n: 942
prevalence: 0.189
mean predicted probability: 0.528
Brier: 0.2506
Brier skill: -0.6355
ROC AUC: 0.7635
PR AUC: 0.4361

=== BRIER / CALIBRATION SUMMARY ===
      outcome   n  positives  negatives  prevalence  mean_predicted_probability  mean_probability_positives  mean_probability_negatives  brier_score  brier_reference_prevalence_model  brier_skill_score